# Databricks / PySpark dla Data Analyst/Engineer — Moduł 15: Projekt końcowy

To ostatni moduł kursu — pełny, wykonywalny **potok danych od początku do końca**,
łączący wszystko, czego nauczyliśmy się w Modułach 1-14: wczytanie surowych, "brudnych"
danych, ich oczyszczenie i przekształcenie DataFrame API, wzbogacenie joinami i funkcjami
okienkowymi, analizę Spark SQL, wytrenowanie i ocenę modelu MLlib oraz zapis wyników jako
partycjonowany Parquet. Całość ułożymy w duchu **architektury Medallion** z Modułu 11:
bronze (surowe dane) → silver (oczyszczone i połączone) → gold (gotowe do analiz i ML).

## Spis treści
1. [Bronze: symulacja "brudnych" surowych danych](#sec1)
2. [Silver: oczyszczanie i transformacje (DataFrame API)](#sec2)
3. [Silver: wzbogacanie — joiny i funkcje okienkowe](#sec3)
4. [Gold: analiza biznesowa (Spark SQL)](#sec4)
5. [Gold: model MLlib — przewidywanie anulowania zamówienia](#sec5)
6. [Zapis wyników — partycjonowany Parquet](#sec6)
7. [Podsumowanie kursu i ćwiczenia końcowe](#sec7)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")


> ⚡ **Dlaczego zmniejszamy shuffle partitions**
>
> Jak poznaliśmy w Modułach 9 i 13, domyślne 200 partycji shuffle'a to duży narzut na małym, lokalnym zbiorze danych. Ustawiamy 8 na początek CAŁEGO notebooka -- w prawdziwym środowisku produkcyjnym (Databricks, duży klaster) zostawilibyśmy domyślną wartość albo dopasowali ją do rozmiaru klastra.

<a id="sec1"></a>
## 1. Bronze: symulacja "brudnych" surowych danych

W prawdziwym projekcie dane wejściowe rzadko są idealne — bywają duplikaty, brakujące
wartości, niespójne wielkości liter. Zasymulujmy to: wygenerujemy nasz znany zestaw danych
sklepu internetowego, celowo go "zabrudzimy", zapiszemy jako surowe pliki CSV (warstwa
**bronze**), a potem wczytamy je z powrotem — dokładnie tak, jak wyglądałby prawdziwy
proces ingestii danych.

In [ ]:
import numpy as np
import pandas as pd
import os

rng = np.random.default_rng(42)

# --- generujemy "czyste" dane, tak jak w calym kursie -----------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        })
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- teraz celowo "brudzimy" dane, jakby pochodzily z prawdziwego zrodla ----
klienci_brudne = klienci_pd.copy()
# niespojne wielkosci liter w miastach
maska_wielkosc = rng.random(len(klienci_brudne)) < 0.15
klienci_brudne.loc[maska_wielkosc, "miasto"] = klienci_brudne.loc[maska_wielkosc, "miasto"].str.upper()
# duplikaty wierszy (typowy blad przy wielokrotnej ingestii)
duplikaty = klienci_brudne.sample(10, random_state=1)
klienci_brudne = pd.concat([klienci_brudne, duplikaty], ignore_index=True)
# brakujace wartosci w segmencie
maska_brak = rng.random(len(klienci_brudne)) < 0.05
klienci_brudne.loc[maska_brak, "segment"] = None

pozycje_brudne = pozycje_zamowien_pd.copy()
# kilka bledych (ujemnych) ilosci -- blad systemu zrodlowego
maska_ujemna = rng.random(len(pozycje_brudne)) < 0.01
pozycje_brudne.loc[maska_ujemna, "ilosc"] = -pozycje_brudne.loc[maska_ujemna, "ilosc"]

os.makedirs("bronze", exist_ok=True)
klienci_brudne.to_csv("bronze/klienci_raw.csv", index=False)
produkty_pd.to_csv("bronze/produkty_raw.csv", index=False)
zamowienia_pd.to_csv("bronze/zamowienia_raw.csv", index=False)
pozycje_brudne.to_csv("bronze/pozycje_zamowien_raw.csv", index=False)

print(f"Zapisano surowe pliki CSV. Klienci (z duplikatami): {len(klienci_brudne)} wierszy (oryginalnie {n_klienci}).")


In [ ]:
# Wczytujemy surowe dane, tak jak w prawdziwej ingestii -- z jawnym schematem
# (poznanym w Module 6), zeby uniknac zgadywania typow przez Sparka
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schemat_klienci = StructType([
    StructField("klient_id", IntegerType(), False),
    StructField("miasto", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("dni_od_rejestracji", IntegerType(), True),
])

klienci_bronze = spark.read.csv("bronze/klienci_raw.csv", header=True, schema=schemat_klienci)
produkty_bronze = spark.read.csv("bronze/produkty_raw.csv", header=True, inferSchema=True)
zamowienia_bronze = spark.read.csv("bronze/zamowienia_raw.csv", header=True, inferSchema=True)
pozycje_bronze = spark.read.csv("bronze/pozycje_zamowien_raw.csv", header=True, inferSchema=True)

print(f"klienci_bronze: {klienci_bronze.count()} wierszy")
klienci_bronze.show(5)


<a id="sec2"></a>
## 2. Silver: oczyszczanie i transformacje (DataFrame API)

Teraz stosujemy dokładnie te operacje DataFrame API, które poznaliśmy w Modułach 2-3:
usuwanie duplikatów, obsługę braków, standaryzację tekstu i filtrowanie nieprawidłowych
wierszy.

In [ ]:
from pyspark.sql import functions as F

klienci_silver = (
    klienci_bronze
    .dropDuplicates(["klient_id"])                                   # usuwamy duplikaty wierszy
    .withColumn("miasto", F.initcap(F.lower(F.col("miasto"))))        # standaryzacja wielkosci liter
    .fillna({"segment": "Standardowy"})                                # rozsadna wartosc domyslna dla brakow
)

print(f"Po oczyszczeniu: {klienci_silver.count()} wierszy (bylo {klienci_bronze.count()} przed usunieciem duplikatow).")
klienci_silver.groupBy("miasto").count().orderBy("miasto").show()


In [ ]:
# Filtrujemy nieprawidlowe pozycje zamowien (ujemna ilosc = blad zrodla danych)
liczba_przed = pozycje_bronze.count()
pozycje_silver = pozycje_bronze.filter(F.col("ilosc") > 0)
liczba_po = pozycje_silver.count()

print(f"Odrzucono {liczba_przed - liczba_po} nieprawidlowych pozycji (ujemna ilosc).")


> ⚠️ **Zawsze mierz, co odrzucasz**
>
> W prawdziwym potoku danych każdy krok czyszczenia powinien logować, ILE wierszy odrzucił lub zmienił -- inaczej cichy spadek jakości danych (np. system źródłowy zaczyna wysyłać 30% błędnych rekordów) przejdzie niezauważony. Powyższe `print()` to uproszczona wersja takiego monitoringu jakości danych.

<a id="sec3"></a>
## 3. Silver: wzbogacanie — joiny i funkcje okienkowe

Łączymy oczyszczone tabele (Moduł 4) i dodajemy kolumny wyliczane funkcjami okienkowymi
(Moduł 7) -- w tym przypadku łączną wartość zamówienia oraz numer zamówienia klienta w
kolejności chronologicznej (przydatny sygnał: "czy to pierwsze zamówienie klienta?").

In [ ]:
from pyspark.sql import Window

wartosc_zamowien = (
    pozycje_silver.groupBy("zamowienie_id")
    .agg(
        F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"),
        F.count("*").alias("liczba_pozycji"),
        F.avg("cena_jednostkowa").alias("srednia_cena"),
    )
)

zamowienia_silver = (
    zamowienia_bronze.join(wartosc_zamowien, "zamowienie_id", "inner")
    .join(klienci_silver, "klient_id", "inner")
    .withColumn("czy_anulowane", (F.col("status") == "Anulowane").cast("int"))
)

okno_klienta = Window.partitionBy("klient_id").orderBy("data_zamowienia")
zamowienia_silver = zamowienia_silver.withColumn(
    "numer_zamowienia_klienta", F.row_number().over(okno_klienta)
)

zamowienia_silver.select(
    "zamowienie_id", "klient_id", "segment", "wartosc", "numer_zamowienia_klienta", "czy_anulowane"
).show(10)


In [ ]:
zamowienia_silver.cache()
print(f"zamowienia_silver: {zamowienia_silver.count()} wierszy (po cache -- kolejne odwolania beda szybsze, Modul 9).")


<a id="sec4"></a>
## 4. Gold: analiza biznesowa (Spark SQL)

Rejestrujemy wzbogaconą tabelę jako widok tymczasowy (Moduł 5) i odpowiadamy czystym
SQL-em na konkretne pytania biznesowe -- dokładnie tak, jak zrobiłby to analityk, który
nie musi znać składni DataFrame API.

In [ ]:
zamowienia_silver.createOrReplaceTempView("zamowienia_view")

raport_segmenty = spark.sql("""
    SELECT
        segment,
        COUNT(*) AS liczba_zamowien,
        ROUND(AVG(wartosc), 2) AS srednia_wartosc,
        ROUND(100.0 * SUM(czy_anulowane) / COUNT(*), 1) AS procent_anulowan
    FROM zamowienia_view
    GROUP BY segment
    ORDER BY srednia_wartosc DESC
""")
raport_segmenty.show()


In [ ]:
raport_miasta = spark.sql("""
    SELECT
        miasto,
        COUNT(*) AS liczba_zamowien,
        ROUND(SUM(wartosc), 2) AS suma_przychodow
    FROM zamowienia_view
    WHERE status != 'Anulowane'
    GROUP BY miasto
    ORDER BY suma_przychodow DESC
""")
raport_miasta.show()


> ⚡ **DataFrame API czy Spark SQL -- w tym module obydwa**
>
> Sekcja 3 użyła DataFrame API (joiny, window functions), a sekcja 4 -- czystego SQL-a na tym samym, wzbogaconym DataFrame. To najlepsza praktyka z prawdziwych projektów: transformacje/inżynieria cech w DataFrame API, raporty biznesowe i eksploracja w SQL -- oba API współdzielą ten sam silnik wykonawczy (Moduł 5).

<a id="sec5"></a>
## 5. Gold: model MLlib — przewidywanie anulowania zamówienia

Trenujemy model klasyfikacji binarnej (Moduły 12-13), który przewiduje, czy zamówienie
zostanie anulowane, na podstawie cech dostępnych W MOMENCIE złożenia zamówienia (segment
klienta, miasto, liczba pozycji, średnia cena, numer kolejnego zamówienia klienta) --
świadomie NIE używamy tu `status` ani samej `wartosc` jako cechy wejściowej, żeby uniknąć
wycieku informacji z przyszłości (*data leakage*).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

train, test = zamowienia_silver.randomSplit([0.8, 0.2], seed=42)

indexer_segment = StringIndexer(inputCol="segment", outputCol="segment_idx", handleInvalid="keep")
indexer_miasto = StringIndexer(inputCol="miasto", outputCol="miasto_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["segment_idx", "miasto_idx"], outputCols=["segment_ohe", "miasto_ohe"])

assembler = VectorAssembler(
    inputCols=["segment_ohe", "miasto_ohe", "liczba_pozycji", "srednia_cena", "numer_zamowienia_klienta"],
    outputCol="features",
)
model_lr = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")

pipeline = Pipeline(stages=[indexer_segment, indexer_miasto, encoder, assembler, model_lr])
model = pipeline.fit(train)

predykcje = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="czy_anulowane", metricName="areaUnderROC")
auc = evaluator.evaluate(predykcje)
print(f"AUC-ROC na zbiorze testowym: {auc:.3f}")


> 🧱 **To ten sam wzorzec co w Module 12-13**
>
> Pipeline (StringIndexer -> OneHotEncoder -> VectorAssembler -> model), podział train/test i BinaryClassificationEvaluator to dokładnie te same klocki co w Modułach 12-13 -- tutaj po prostu składamy je w kontekście pełnego, rzeczywistego potoku danych, zamiast izolowanego przykładu.

<a id="sec6"></a>
## 6. Zapis wyników — partycjonowany Parquet

Na koniec zapisujemy DWIE tabele wynikowe do warstwy **gold** jako Parquet partycjonowany
po `segment` (Moduł 6) — format kolumnowy gotowy do szybkich analiz w narzędziach BI albo
kolejnych notebookach: (1) predykcje modelu, (2) raport segmentowy z sekcji 4.

In [ ]:
predykcje.select(
    "zamowienie_id", "klient_id", "segment", "miasto", "wartosc", "czy_anulowane", "prediction"
).write.mode("overwrite").partitionBy("segment").parquet("gold/predykcje_anulowania")

raport_segmenty.write.mode("overwrite").parquet("gold/raport_segmenty")

print("Zapisano wyniki do gold/predykcje_anulowania (partycjonowane po segment) i gold/raport_segmenty.")


In [ ]:
# Weryfikacja: odczyt z partition pruning -- Spark czyta TYLKO partycje "Premium"
sprawdzenie = spark.read.parquet("gold/predykcje_anulowania").filter(F.col("segment") == "Premium")
sprawdzenie.explain()
print(f"Liczba wierszy w partycji Premium: {sprawdzenie.count()}")


<a id="sec7"></a>
## 7. Podsumowanie kursu i ćwiczenia końcowe

Zbudowaliśmy kompletny potok danych w duchu architektury Medallion:

- **bronze** -- surowe, "brudne" dane CSV, wczytane z jawnym schematem,
- **silver** -- oczyszczone (duplikaty, braki, standaryzacja), wzbogacone joinami i
  funkcjami okienkowymi, zbuforowane w pamięci,
- **gold** -- raporty biznesowe (Spark SQL), model MLlib (Pipeline + ocena AUC-ROC) oraz
  finalne wyniki zapisane jako partycjonowany Parquet.

To domyka cały kurs Databricks/PySpark -- od `SparkSession` w Module 1, przez
DataFrame API, joiny, Spark SQL, czytanie/zapis danych, funkcje okienkowe, UDF-y,
optymalizację, Delta Lake, architekturę Lakehouse, MLlib, aż po pracę w prawdziwym
Databricks Workspace w Module 14.

### 📝 Ćwiczenie 1: Dodaj kolejną cechę

Dodaj do modelu z sekcji 5 nową cechę: `dni_od_rejestracji` klienta (z tabeli `klienci_silver`, którą trzeba dołączyć do zbioru cech przed treningiem). Sprawdź, czy AUC-ROC się poprawiło.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
assembler2 = VectorAssembler(
    inputCols=["segment_ohe", "miasto_ohe", "liczba_pozycji", "srednia_cena",
               "numer_zamowienia_klienta", "dni_od_rejestracji"],
    outputCol="features",
)
pipeline2 = Pipeline(stages=[indexer_segment, indexer_miasto, encoder, assembler2, model_lr])
model2 = pipeline2.fit(train)
predykcje2 = model2.transform(test)
auc2 = evaluator.evaluate(predykcje2)
print(f"AUC-ROC z dodatkowa cecha: {auc2:.3f} (bylo {auc:.3f})")
```

</details>

### 📝 Ćwiczenie 2: Raport o pracownikach -- rozszerzenie potoku

Zbiór danych w tym module nie zawiera tabeli `pracownicy`, ale w Module 4-5 była obecna. Napisz zapytanie Spark SQL na `zamowienia_view`, które liczy sumę przychodów i liczbę zamówień PER MIESIĄC (użyj `date_format(data_zamowienia, 'yyyy-MM')`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
raport_miesieczny = spark.sql("""
    SELECT
        date_format(data_zamowienia, 'yyyy-MM') AS miesiac,
        COUNT(*) AS liczba_zamowien,
        ROUND(SUM(wartosc), 2) AS suma_przychodow
    FROM zamowienia_view
    WHERE status != 'Anulowane'
    GROUP BY date_format(data_zamowienia, 'yyyy-MM')
    ORDER BY miesiac
""")
raport_miesieczny.show(24)
```

</details>

### 📝 Ćwiczenie 3: Odczytaj zapisane wyniki

Wczytaj z powrotem `gold/raport_segmenty` (zapisany w sekcji 6) jako nowy DataFrame i wyświetl go -- sprawdź, że dane po zapisie i odczycie są identyczne z oryginałem.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
raport_segmenty_odczytany = spark.read.parquet("gold/raport_segmenty")
raport_segmenty_odczytany.orderBy(F.col("srednia_wartosc").desc()).show()
```

</details>

> 🔥 **Wyzwanie końcowe: pełny raport z alertem jakości danych**
>
> Rozbuduj krok czyszczenia (sekcja 2), żeby PRZED odrzuceniem nieprawidłowych pozycji zamówień policzyć procent odrzuconych wierszy. Jeśli przekracza on 5%, wypisz ostrzeżenie (`print("UWAGA: ...")`) -- symulację prostego alertu jakości danych, jaki w prawdziwym potoku wysłałby powiadomienie (Moduł 14, `dbutils` + Jobs). Na koniec wypisz jednym zdaniem podsumowanie CAŁEGO potoku: ile zamówień trafiło do warstwy gold i jakie było końcowe AUC-ROC modelu.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
procent_odrzuconych = 100.0 * (liczba_przed - liczba_po) / liczba_przed
if procent_odrzuconych > 5.0:
    print(f"UWAGA: odrzucono {procent_odrzuconych:.1f}% pozycji zamowien -- sprawdz zrodlo danych!")
else:
    print(f"Jakosc danych OK: odrzucono tylko {procent_odrzuconych:.2f}% pozycji zamowien.")

print(
    f"Podsumowanie potoku: {zamowienia_silver.count()} zamowien w warstwie gold, "
    f"koncowe AUC-ROC modelu = {auc:.3f}."
)
```

W prawdziwym Databricks ten alert wyslalby powiadomienie przez integracje z Jobs (Modul 14) zamiast print() -- np. zatrzymujac potok i wysylajac e-mail/Slack, gdy jakosc danych spadnie ponizej progu.

</details>

## Gratulacje!

To koniec kursu Databricks / PySpark -- i całej serii kursów zaczętej od Pythona, przez
SQL/PostgreSQL i Statystykę. Przeszedłeś/przeszłaś drogę od pojedynczych zapytań SQL i
`pandas.DataFrame`, przez testy statystyczne i regresję, aż po rozproszone przetwarzanie
petabajtów danych i uczenie maszynowe na skalę klastra. Powodzenia w prawdziwych
projektach danych!